# Multiprocess unified-unregister wire E2E

This Run-All notebook starts the real in-process ZMQ `MessageQueueServer`, connects concurrent `MessageQueueClient` instances, and alternates both public unregister request types. Three independent state owners model KV, Q-ring, and Blend state. Assertions prove that every request receives a response, every owner is notified, all state is released, and duplicate unregisters do not release the same state twice.

The exercised lifecycle path is CPU control-plane code; no GPU or model download is required. Run all cells from an LMCache checkout after installing the test dependencies with `uv pip install -r requirements/test.txt`.

In [ ]:
# SPDX-License-Identifier: Apache-2.0
# Standard
from importlib.metadata import version
from pathlib import Path
import json
import os
import platform
import shlex
import subprocess
import sys

repo_candidates = (Path.cwd(), *Path.cwd().parents)
repo_root = next(
    (path for path in repo_candidates if (path / "pyproject.toml").is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("run from the LMCache repository")
script = repo_root / "benchmarks/microbenchmark/mp_unregister_lifecycle_e2e.py"
command = [
    sys.executable,
    str(script),
    "--instances",
    "512",
    "--workers",
    "16",
    "--duplicates",
    "4",
]
print("python:", platform.python_version())
print("pyzmq:", version("pyzmq"))
print("$", shlex.join(command))
env = {**os.environ, "LMCACHE_LOG_LEVEL": "WARNING"}
completed = subprocess.run(
    command,
    check=False,
    capture_output=True,
    text=True,
    cwd=repo_root,
    env=env,
)
if completed.stderr:
    print(completed.stderr, file=sys.stderr)
if completed.returncode != 0:
    raise RuntimeError(f"E2E exited with {completed.returncode}")
print(completed.stdout)
json_start = completed.stdout.find("{")
if json_start < 0:
    raise ValueError("E2E output did not contain JSON evidence")
evidence = json.loads(completed.stdout[json_start:])
assert evidence["workload"] == {
    "instances": 512,
    "clients": 16,
    "requests_per_instance": 4,
    "requests": 2048,
}
assert all(evidence["invariants"].values())
for owner in evidence["owners"].values():
    assert owner == {
        "remaining": 0,
        "released": 512,
        "unique_released": 512,
        "drop_calls": 2048,
    }
print("Unified unregister wire E2E invariants passed")